# Two-dimensional sliding-window embedding of an AR(1) process

We simulate the centered AR(1) model

$$
X_t = \phi X_{t-1} + \varepsilon_t,
\qquad \varepsilon_t \overset{\mathrm{iid}}{\sim} \operatorname{Unif}(-a,a),
\qquad |\phi|<1,
$$

and form the lag-pair (sliding-window) embedding

$$
Y_t=(X_{t-1},X_t)\in\mathbb R^2.
$$

Directly from the model equation, every embedded point satisfies

$$
-a \le X_t-\phi X_{t-1} \le a.
$$

Thus the point cloud lies in the strip between $y=\phi x-a$ and $y=\phi x+a$. The figure below recreates the supplied sketch with the simulated embedding added. The green segment is the horizontal chord of the strip; the red double arrow is its perpendicular Euclidean width. These are different notions of width.

The support-strip formulas in this notebook are derived directly from the displayed AR(1) equation.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Locate Code/Code_AR_processes whether the notebook is launched from that folder
# or from the repository root.
working_directory = Path.cwd().resolve()
code_directory = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "ar_tools").is_dir()
    ),
    None,
)
if code_directory is None:
    repository_root = next(
        (
            candidate
            for candidate in (working_directory, *working_directory.parents)
            if (candidate / "Code" / "Code_AR_processes" / "ar_tools").is_dir()
        ),
        None,
    )
    if repository_root is None:
        raise RuntimeError("Could not locate Code/Code_AR_processes.")
    code_directory = repository_root / "Code" / "Code_AR_processes"

# Reuse the shared PDF exporter from the KDE utilities.
sys.path.insert(0, str(code_directory.parent / "Code_kde"))
os.environ.setdefault(
    "MPLCONFIGDIR",
    str(code_directory / ".matplotlib-cache"),
)

import matplotlib.pyplot as plt
import numpy as np

from kde_tools.pdf_export import enable_automatic_pdf_saving

# Import the same innovation sampler used by every shared AR simulator.
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))
from ar_tools.innovations import beta_innovation_bounds, sample_beta_innovations

notebook_name = "ar1_sliding_window_embedding"
pdf_output_directory = enable_automatic_pdf_saving(
    notebook_name,
    output_root=code_directory / "outputs" / "pdfs",
)
image_output_directory = (
    code_directory / "outputs" / "images" / notebook_name
)
image_output_directory.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
print(f"Automatic PDF directory: {pdf_output_directory}")

## Simulate the process and build the lag-pair cloud

The seed and all model parameters are explicit, so re-running the notebook reproduces the same point cloud. The burn-in is discarded before the embedding is formed.

In [ ]:
phi = 0.72
innovation_radius = beta_innovation_bounds()[1]  # Uniform baseline; variance 1/3.
sample_size = 700
burn_in = 400
seed = 20260920

if not abs(phi) < 1.0:
    raise ValueError("This notebook assumes a stable AR(1): require |phi| < 1.")
if sample_size < 2 or burn_in < 0:
    raise ValueError("Require sample_size >= 2 and burn_in >= 0.")

rng = np.random.default_rng(seed)
total_steps = burn_in + sample_size
innovations = sample_beta_innovations(
    size=total_steps,
    rng=rng,
)
series = np.empty(total_steps + 1, dtype=float)
series[0] = 0.0
for time_index in range(1, total_steps + 1):
    series[time_index] = (
        phi * series[time_index - 1] + innovations[time_index - 1]
    )

observed_series = series[burn_in:]
embedding = np.column_stack((observed_series[:-1], observed_series[1:]))
embedding_residuals = embedding[:, 1] - phi * embedding[:, 0]

floating_point_tolerance = 10 * np.finfo(float).eps
if np.max(np.abs(embedding_residuals)) > innovation_radius + floating_point_tolerance:
    raise AssertionError("An embedded point fell outside the theoretical strip.")

print(f"Embedded lag pairs: {len(embedding)}")
print(
    "Residual range: "
    f"[{embedding_residuals.min():.4f}, {embedding_residuals.max():.4f}]"
)

## Annotated sliding-window embedding

For the centered model, the blue center line is

$$
H_{\phi,0}=\{(x,y):y=\phi x\}.
$$

The perpendicular distance between the two red support boundaries is

$$
w_\perp=\frac{2a}{\sqrt{1+\phi^2}},
$$

while a horizontal slice through the strip has length $2a/|\phi|$ when $\phi\ne0$. The plot uses the colors and geometric cues of the supplied sketch while keeping these two quantities distinct.

In [ ]:
theoretical_state_bound = innovation_radius / (1.0 - abs(phi))
axis_limit = 1.08 * theoretical_state_bound
x_grid = np.linspace(-axis_limit, axis_limit, 600)
lower_boundary = phi * x_grid - innovation_radius
upper_boundary = phi * x_grid + innovation_radius
center_line = phi * x_grid

boundary_color = "#D7191C"
center_color = "#0B559F"
chord_color = "#00876C"
point_color = "#355C7D"

fig, ax = plt.subplots(figsize=(10.0, 8.0))
fig.suptitle(
    "AR(1) sliding-window embedding with bounded innovations",
    fontsize=17,
    fontweight="bold",
    y=0.98,
)
ax.set_title(
    rf"$X_t={phi:.2f}X_{{t-1}}+\varepsilon_t$, "
    rf"$\varepsilon_t\sim\mathrm{{Unif}}(-{innovation_radius:g},{innovation_radius:g})$",
    fontsize=12,
    pad=12,
)

ax.fill_between(
    x_grid,
    lower_boundary,
    upper_boundary,
    color=boundary_color,
    alpha=0.055,
    zorder=0,
)
ax.scatter(
    embedding[:, 0],
    embedding[:, 1],
    s=18,
    color=point_color,
    alpha=0.34,
    edgecolors="none",
    rasterized=True,
    label=rf"lag pairs $(X_{{t-1}},X_t)$, $n={sample_size}$",
    zorder=2,
)
ax.plot(
    x_grid,
    upper_boundary,
    color=boundary_color,
    linewidth=2.8,
    label=rf"support boundaries $y=\phi x\pm {innovation_radius:g}$",
    zorder=4,
)
ax.plot(
    x_grid,
    lower_boundary,
    color=boundary_color,
    linewidth=2.8,
    zorder=4,
)
ax.plot(
    x_grid,
    center_line,
    color=center_color,
    linewidth=3.0,
    label=rf"center line $H_{{\phi,0}}$: $y={phi:.2f}x$",
    zorder=5,
)

# Green horizontal chord, matching the horizontal segment in the sketch.
slice_y = -0.25
horizontal_intersections = np.sort(
    np.array(
        [
            (slice_y - innovation_radius) / phi,
            (slice_y + innovation_radius) / phi,
        ]
    )
)
ax.plot(
    horizontal_intersections,
    [slice_y, slice_y],
    color=chord_color,
    linewidth=3.5,
    marker="|",
    markersize=16,
    markeredgewidth=3.5,
    zorder=7,
)
ax.text(
    horizontal_intersections.mean(),
    slice_y + 0.22,
    rf"horizontal chord $=2a/|\phi|={2 * innovation_radius / abs(phi):.2f}$",
    color=chord_color,
    fontsize=10.5,
    ha="center",
    va="bottom",
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=1.5),
    zorder=8,
)

# Red double arrow normal to the strip: the intrinsic Euclidean width.
normal_vector = np.array([-phi, 1.0]) / np.sqrt(1.0 + phi**2)
arrow_center = np.array([1.20, phi * 1.20])
half_width = innovation_radius / np.sqrt(1.0 + phi**2)
arrow_start = arrow_center - half_width * normal_vector
arrow_end = arrow_center + half_width * normal_vector
ax.annotate(
    "",
    xy=arrow_end,
    xytext=arrow_start,
    arrowprops=dict(
        arrowstyle="<->",
        color=boundary_color,
        linewidth=2.8,
        shrinkA=0,
        shrinkB=0,
    ),
    zorder=9,
)
ax.text(
    arrow_center[0] + 0.28,
    arrow_center[1] + 0.05,
    rf"$w_\perp=\dfrac{{2a}}{{\sqrt{{1+\phi^2}}}}={2 * half_width:.2f}$",
    color=boundary_color,
    fontsize=11,
    ha="left",
    va="center",
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=1.5),
    zorder=10,
)

line_angle_degrees = np.degrees(np.arctan(phi))
center_label_x = 0.60 * axis_limit
ax.text(
    center_label_x,
    phi * center_label_x + 0.10,
    r"$H_{\phi,0}$",
    color=center_color,
    fontsize=14,
    fontweight="bold",
    rotation=line_angle_degrees,
    rotation_mode="anchor",
    ha="left",
    va="bottom",
    zorder=10,
)

ax.set(
    xlim=(-axis_limit, axis_limit),
    ylim=(-axis_limit, axis_limit),
    xlabel=r"$X_{t-1}$",
    ylabel=r"$X_t$",
)
ax.set_aspect("equal", adjustable="box")
ax.axhline(0.0, color="#9AA5B1", linewidth=0.8, zorder=1)
ax.axvline(0.0, color="#9AA5B1", linewidth=0.8, zorder=1)
ax.grid(True, color="#D9E2EC", linewidth=0.8, alpha=0.75)
ax.legend(
    loc="upper left",
    frameon=True,
    framealpha=0.96,
    facecolor="white",
    edgecolor="#BCCCDC",
    fontsize=9.5,
)
for spine in ax.spines.values():
    spine.set_color("#829AB1")

fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.95))
png_output_path = image_output_directory / "ar1_sliding_window_embedding.png"
fig.savefig(png_output_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"Saved PNG: {png_output_path}")

## Reading the figure

- Each dot is one overlapping length-two window $(X_{t-1},X_t)$.
- The red lines are the exact innovation-support constraints.
- The blue line is the zero-innovation relation.
- The green chord reproduces the horizontal measurement in the sketch.
- The red double arrow shows the coordinate-free perpendicular width of the slab.

The innovation radius is fixed by the mean-zero, variance-1/3 normalization.
Change `phi`, `sample_size`, or `seed` in the simulation cell and re-run the notebook to generate another reproducible embedding.